**420-A58-SF - Algorithmes d'apprentissage non supervisé - Hiver 2026 - Spécialisation technique en Intelligence Artificielle**<br/>
MIT License - Copyright (c) 2026 Mikaël Swawola
<br/>
**Objectif:** cette séance de travaux pratiques a pour objectif l'évaluation de la qualité du partitionnement à l'aide de métriques internes : **Score de Silhouette**, **Indice de Davies-Bouldin** et **Score de Calinski-Harabasz**

In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sns.set()
sns.set(style="darkgrid")
sns.set_context("notebook", font_scale=1.5, rc={"lines.linewidth": 2.5})
plt.rcParams['figure.figsize'] = (12, 8)

## 1 - Génération et visualisation des données

**Exercice 1-1 - Générer un jeu de données synthétique avec 4 clusters bien séparés**

Utiliser la fonction `make_blobs` de `sklearn.datasets` avec les paramètres suivants :
* n_samples = 400
* n_features = 2
* centers = 4
* cluster_std = 1.0
* random_state = 42

In [ ]:
from sklearn.datasets import make_blobs

X, y_true = make_blobs(n_samples=400, n_features=2, centers=4, 
                       cluster_std=1.0, random_state=42)

**Exercice 1-2 - Créer un DataFrame pandas avec les données et visualiser les points (sans afficher les vraies étiquettes)**

In [ ]:
df = pd.DataFrame(X, columns=['x0', 'x1'])

plt.figure(figsize=(10, 6))
sns.scatterplot(x='x0', y='x1', data=df, s=100, alpha=0.7)
plt.title('Données synthétiques - 400 observations')
plt.show()

## 2 - Partitionnement K-moyennes avec différentes valeurs de K

**Exercice 2-1 - Importer KMeans et StandardScaler de scikit-learn**

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

**Exercice 2-2 - Standardiser les données**

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
df_scaled = pd.DataFrame(X_scaled, columns=['x0', 'x1'])

**Exercice 2-3 - Effectuer le partitionnement K-moyennes pour K variant de 2 à 8**

Stocker les modèles dans un dictionnaire avec K comme clé

In [ ]:
models = {}
k_values = range(2, 9)

for k in k_values:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    kmeans.fit(X_scaled)
    models[k] = kmeans

**Exercice 2-4 - Visualiser les résultats pour K=3, K=4 et K=5 sur trois sous-graphiques**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, k in enumerate([3, 4, 5]):
    ax = axes[idx]
    labels = models[k].labels_
    
    df_plot = df_scaled.copy()
    df_plot['cluster'] = labels
    
    sns.scatterplot(x='x0', y='x1', hue='cluster', data=df_plot, 
                   palette='Set2', s=100, alpha=0.7, ax=ax, legend='full')
    
    # Afficher les centroïdes
    centroids = models[k].cluster_centers_
    ax.scatter(centroids[:, 0], centroids[:, 1], 
              marker='X', s=300, c='red', edgecolors='black', linewidths=2)
    
    ax.set_title(f'K-moyennes avec K={k}')
    ax.set_xlabel('x0 (standardisé)')
    ax.set_ylabel('x1 (standardisé)')

plt.tight_layout()
plt.show()

## 3 - Score de Silhouette

Le **score de Silhouette** mesure la cohésion intra-cluster et la séparation inter-cluster. Il varie entre -1 et 1 :

$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$

Où :
* $a(i)$ : distance moyenne entre l'observation $i$ et les autres observations du même cluster
* $b(i)$ : distance moyenne entre l'observation $i$ et les observations du cluster le plus proche

**Interprétation :**
* Score proche de **+1** : l'observation est bien assignée à son cluster
* Score proche de **0** : l'observation est proche de la frontière entre deux clusters
* Score proche de **-1** : l'observation est probablement mal assignée

**Exercice 3-1 - Importer silhouette_score de sklearn.metrics**

In [ ]:
from sklearn.metrics import silhouette_score

**Exercice 3-2 - Calculer le score de Silhouette pour chaque valeur de K (de 2 à 8)**

In [ ]:
silhouette_scores = {}

for k in k_values:
    labels = models[k].labels_
    score = silhouette_score(X_scaled, labels)
    silhouette_scores[k] = score

**Exercice 3-3 - Afficher les scores sous forme de tableau et identifier la valeur optimale de K**

In [ ]:
df_silhouette = pd.DataFrame(list(silhouette_scores.items()), 
                             columns=['K', 'Score Silhouette'])
print(df_silhouette)
print(f"\nK optimal (Silhouette) : {df_silhouette.loc[df_silhouette['Score Silhouette'].idxmax(), 'K']}")

**Exercice 3-4 - Tracer un graphique du score de Silhouette en fonction de K**

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(list(silhouette_scores.keys()), list(silhouette_scores.values()), 
         marker='o', linewidth=2, markersize=10)
plt.xlabel('Nombre de clusters (K)')
plt.ylabel('Score de Silhouette')
plt.title('Score de Silhouette en fonction de K')
plt.grid(True, alpha=0.3)
plt.xticks(k_values)
plt.show()

## 4 - Indice de Davies-Bouldin

L'**indice de Davies-Bouldin** mesure le ratio moyen entre la dispersion intra-cluster et la séparation inter-cluster.

$$DB = \frac{1}{k} \sum_{i=1}^{k} \max_{j \neq i} \left( \frac{s_i + s_j}{d_{ij}} \right)$$

Où :
* $s_i$ : dispersion moyenne du cluster $i$
* $d_{ij}$ : distance entre les centroïdes des clusters $i$ et $j$

**Interprétation :**
* **Plus la valeur est faible, meilleur est le partitionnement**
* Une valeur proche de 0 indique des clusters bien séparés et compacts

**Exercice 4-1 - Importer davies_bouldin_score de sklearn.metrics**

In [ ]:
from sklearn.metrics import davies_bouldin_score

**Exercice 4-2 - Calculer l'indice de Davies-Bouldin pour chaque valeur de K (de 2 à 8)**

In [ ]:
davies_bouldin_scores = {}

for k in k_values:
    labels = models[k].labels_
    score = davies_bouldin_score(X_scaled, labels)
    davies_bouldin_scores[k] = score

**Exercice 4-3 - Afficher les scores et identifier la valeur optimale de K**

In [ ]:
df_db = pd.DataFrame(list(davies_bouldin_scores.items()), 
                     columns=['K', 'Indice Davies-Bouldin'])
print(df_db)
print(f"\nK optimal (Davies-Bouldin) : {df_db.loc[df_db['Indice Davies-Bouldin'].idxmin(), 'K']}")

**Exercice 4-4 - Tracer un graphique de l'indice de Davies-Bouldin en fonction de K**

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(list(davies_bouldin_scores.keys()), list(davies_bouldin_scores.values()), 
         marker='o', linewidth=2, markersize=10, color='orange')
plt.xlabel('Nombre de clusters (K)')
plt.ylabel('Indice de Davies-Bouldin')
plt.title('Indice de Davies-Bouldin en fonction de K (plus bas = meilleur)')
plt.grid(True, alpha=0.3)
plt.xticks(k_values)
plt.show()

## 5 - Score de Calinski-Harabasz

Le **score de Calinski-Harabasz** (aussi appelé Variance Ratio Criterion) évalue le rapport entre la dispersion inter-cluster et la dispersion intra-cluster.

$$CH = \frac{SS_B / (k-1)}{SS_W / (n-k)}$$

Où :
* $SS_B$ : somme des carrés inter-cluster (between-cluster sum of squares)
* $SS_W$ : somme des carrés intra-cluster (within-cluster sum of squares)
* $k$ : nombre de clusters
* $n$ : nombre d'observations

**Interprétation :**
* **Plus la valeur est élevée, meilleur est le partitionnement**
* Indique des clusters denses et bien séparés

**Exercice 5-1 - Importer calinski_harabasz_score de sklearn.metrics**

In [ ]:
from sklearn.metrics import calinski_harabasz_score

**Exercice 5-2 - Calculer le score de Calinski-Harabasz pour chaque valeur de K (de 2 à 8)**

In [ ]:
calinski_harabasz_scores = {}

for k in k_values:
    labels = models[k].labels_
    score = calinski_harabasz_score(X_scaled, labels)
    calinski_harabasz_scores[k] = score

**Exercice 5-3 - Afficher les scores et identifier la valeur optimale de K**

In [ ]:
df_ch = pd.DataFrame(list(calinski_harabasz_scores.items()), 
                     columns=['K', 'Score Calinski-Harabasz'])
print(df_ch)
print(f"\nK optimal (Calinski-Harabasz) : {df_ch.loc[df_ch['Score Calinski-Harabasz'].idxmax(), 'K']}")

**Exercice 5-4 - Tracer un graphique du score de Calinski-Harabasz en fonction de K**

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(list(calinski_harabasz_scores.keys()), list(calinski_harabasz_scores.values()), 
         marker='o', linewidth=2, markersize=10, color='green')
plt.xlabel('Nombre de clusters (K)')
plt.ylabel('Score de Calinski-Harabasz')
plt.title('Score de Calinski-Harabasz en fonction de K (plus haut = meilleur)')
plt.grid(True, alpha=0.3)
plt.xticks(k_values)
plt.show()

## 6 - Comparaison des métriques

**Exercice 6-1 - Créer un DataFrame récapitulatif avec les trois métriques pour chaque valeur de K**

In [ ]:
df_comparison = pd.DataFrame({
    'K': k_values,
    'Silhouette': silhouette_scores.values(),
    'Davies-Bouldin': davies_bouldin_scores.values(),
    'Calinski-Harabasz': calinski_harabasz_scores.values()
})

print(df_comparison)

**Exercice 6-2 - Tracer les trois métriques sur un même graphique (avec trois sous-graphiques)**

<details>
<summary>
    <font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font>
</summary>
<p>
Utiliser <code>plt.subplots(1, 3, figsize=(18, 5))</code> pour créer trois graphiques côte à côte
</p>
</details>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Score de Silhouette
axes[0].plot(df_comparison['K'], df_comparison['Silhouette'], 
            marker='o', linewidth=2, markersize=10, color='blue')
axes[0].set_xlabel('Nombre de clusters (K)')
axes[0].set_ylabel('Score de Silhouette')
axes[0].set_title('Score de Silhouette\n(plus haut = meilleur)')
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(k_values)

# Indice de Davies-Bouldin
axes[1].plot(df_comparison['K'], df_comparison['Davies-Bouldin'], 
            marker='o', linewidth=2, markersize=10, color='orange')
axes[1].set_xlabel('Nombre de clusters (K)')
axes[1].set_ylabel('Indice de Davies-Bouldin')
axes[1].set_title('Indice de Davies-Bouldin\n(plus bas = meilleur)')
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(k_values)

# Score de Calinski-Harabasz
axes[2].plot(df_comparison['K'], df_comparison['Calinski-Harabasz'], 
            marker='o', linewidth=2, markersize=10, color='green')
axes[2].set_xlabel('Nombre de clusters (K)')
axes[2].set_ylabel('Score de Calinski-Harabasz')
axes[2].set_title('Score de Calinski-Harabasz\n(plus haut = meilleur)')
axes[2].grid(True, alpha=0.3)
axes[2].set_xticks(k_values)

plt.tight_layout()
plt.show()

**Exercice 6-3 - Quelle est la valeur optimale de K selon chaque métrique ? Les résultats sont-ils cohérents ?**

**Réponse :**

* Score de Silhouette : K = 4 (maximum)
* Indice de Davies-Bouldin : K = 4 (minimum)
* Score de Calinski-Harabasz : K = 4 (maximum)
* Conclusion : Les trois métriques sont cohérentes et suggèrent K=4 comme nombre optimal de clusters, ce qui correspond effectivement au nombre de clusters utilisés pour générer les données synthétiques. Cela démontre que ces métriques sont efficaces pour identifier la structure naturelle des données.

## 7 - Test avec des clusters mal séparés

**Exercice 7 - Générer un nouveau jeu de données avec cluster_std=3.0 (clusters qui se chevauchent) et refaire l'analyse. Comment se comportent les métriques ?**

In [ ]:
# Génération des données avec clusters qui se chevauchent
X_overlap, y_overlap = make_blobs(n_samples=400, n_features=2, centers=4, 
                                   cluster_std=3.0, random_state=42)

# Standardisation
X_overlap_scaled = scaler.fit_transform(X_overlap)

# Visualisation
df_overlap = pd.DataFrame(X_overlap_scaled, columns=['x0', 'x1'])
plt.figure(figsize=(10, 6))
sns.scatterplot(x='x0', y='x1', data=df_overlap, s=100, alpha=0.7)
plt.title('Données avec clusters qui se chevauchent')
plt.show()

# Partitionnement pour différentes valeurs de K
models_overlap = {}
silhouette_overlap = {}
davies_bouldin_overlap = {}
calinski_harabasz_overlap = {}

for k in k_values:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    kmeans.fit(X_overlap_scaled)
    models_overlap[k] = kmeans
    
    labels = kmeans.labels_
    silhouette_overlap[k] = silhouette_score(X_overlap_scaled, labels)
    davies_bouldin_overlap[k] = davies_bouldin_score(X_overlap_scaled, labels)
    calinski_harabasz_overlap[k] = calinski_harabasz_score(X_overlap_scaled, labels)

# Comparaison graphique
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(k_values, [silhouette_overlap[k] for k in k_values], 
            marker='o', linewidth=2, label='Chevauchement', color='red')
axes[0].plot(k_values, [silhouette_scores[k] for k in k_values], 
            marker='s', linewidth=2, label='Bien séparés', color='blue', linestyle='--')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Score de Silhouette')
axes[0].set_title('Comparaison Silhouette')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(k_values, [davies_bouldin_overlap[k] for k in k_values], 
            marker='o', linewidth=2, label='Chevauchement', color='red')
axes[1].plot(k_values, [davies_bouldin_scores[k] for k in k_values], 
            marker='s', linewidth=2, label='Bien séparés', color='orange', linestyle='--')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Davies-Bouldin')
axes[1].set_title('Comparaison Davies-Bouldin')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(k_values, [calinski_harabasz_overlap[k] for k in k_values], 
            marker='o', linewidth=2, label='Chevauchement', color='red')
axes[2].plot(k_values, [calinski_harabasz_scores[k] for k in k_values], 
            marker='s', linewidth=2, label='Bien séparés', color='green', linestyle='--')
axes[2].set_xlabel('K')
axes[2].set_ylabel('Calinski-Harabasz')
axes[2].set_title('Comparaison Calinski-Harabasz')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Observation : Avec des clusters qui se chevauchent, les scores sont généralement moins bons")
print("(Silhouette plus bas, Davies-Bouldin plus haut) et le choix optimal de K devient moins évident.")

## Résumé des métriques d'évaluation

| Métrique | Formule | Plage | Optimal | Avantages | Inconvénients |
|----------|---------|-------|---------|-----------|---------------|
| **Silhouette** | $s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$ | [-1, 1] | Maximum (proche de 1) | Intuitive, analyse par échantillon possible | Coûteuse en calcul |
| **Davies-Bouldin** | $DB = \frac{1}{k} \sum_{i=1}^{k} \max_{j \neq i} \left( \frac{s_i + s_j}{d_{ij}} \right)$ | [0, ∞] | Minimum (proche de 0) | Rapide à calculer | Sensible au nombre de clusters |
| **Calinski-Harabasz** | $CH = \frac{SS_B / (k-1)}{SS_W / (n-k)}$ | [0, ∞] | Maximum (valeur élevée) | Rapide, interprétation claire | Favorise les clusters sphériques |

**Fin de l'atelier 01-04 - Solutions**